# ⏳ Notebook 3: Long Polling

Long polling is the easiest way to achieve **near real-time** updates while still using standard HTTP. The server holds the request open until new data is available!

## Learning Objectives

By the end of this notebook, you'll understand:
- How long polling differs from simple polling
- The latency trade-offs of long polling
- How to implement a long polling server and client
- When to choose long polling over other methods

## 🤔 What is Long Polling?

With **simple polling**, the server responds immediately (even if there's nothing new).

With **long polling**, the server **waits** until there's new data before responding!

```
Simple Polling:                    Long Polling:
                                   
Client ─► Server                   Client ─► Server
Client ◄─ "nothing"  (immediate)   Client    ...waiting...
                                   Client    ...waiting...
Client ─► Server                   Client    ...waiting...
Client ◄─ "nothing"  (immediate)   Client ◄─ "new data!"  (when available)
                                   
Client ─► Server                   Client ─► Server
Client ◄─ "new data!"              Client    ...waiting...
```

The key insight: **the server holds the connection open** until it has something to say!

## 📊 Comparing Latency

Let's understand why long polling has lower latency:

```
Simple Polling (2s interval):
────────────────────────────────────────────────────►
     │Poll     │Poll     │Poll     │Poll
     ▼         ▼         ▼         ▼
                    💬 Message arrives here
                              │
                              └─► User sees it HERE (up to 2s later!)

Long Polling:
────────────────────────────────────────────────────►
     │Request held open.....................│
                    💬 Message arrives here
                    │
                    └─► User sees it IMMEDIATELY!
```

## 🛠️ Let's Build It!

### Step 1: Start the Server

Before continuing, start the server in a terminal:

```bash
cd patterns/real-time-updates/servers
python long_polling_server.py
```

You should see: `🚀 Starting Long Polling Server on port 5002`

In [ ]:
# Verify the server is running
import requests

try:
    response = requests.get("http://localhost:5002/health", timeout=2)
    if response.status_code == 200:
        print("✅ Long polling server is running!")
except requests.exceptions.ConnectionError:
    print("❌ Server is not running!")
    print("   Please start it with: python ../servers/long_polling_server.py")

### Step 2: Create the Long Polling Client

In [ ]:
import requests
import time
from datetime import datetime
import threading

class LongPollingClient:
    """
    A long polling client that holds requests open until new data arrives.
    """
    
    def __init__(self, server_url: str, timeout: int = 35):
        self.server_url = server_url
        self.timeout = timeout  # Slightly longer than server timeout
        self.last_timestamp = 0
        self.running = False
        self.session = requests.Session()  # Reuse connections!
    
    def long_poll_once(self):
        """
        Make a single long poll request.
        This blocks until the server responds!
        """
        try:
            start_time = time.time()
            
            response = self.session.get(
                f"{self.server_url}/messages/poll",
                params={"since": self.last_timestamp},
                timeout=self.timeout
            )
            
            elapsed = time.time() - start_time
            
            if response.status_code == 200:
                data = response.json()
                messages = data.get("messages", [])
                timeout = data.get("timeout", False)
                
                if messages:
                    self.last_timestamp = max(msg["timestamp"] for msg in messages)
                
                return {
                    "messages": messages,
                    "timeout": timeout,
                    "wait_time": elapsed
                }
                
        except requests.exceptions.Timeout:
            return {"messages": [], "timeout": True, "wait_time": self.timeout}
        except requests.exceptions.RequestException as e:
            print(f"❌ Request failed: {e}")
            return {"messages": [], "error": str(e)}
    
    def send_message(self, user: str, text: str):
        """
        Send a message to the chat.
        """
        try:
            response = self.session.post(
                f"{self.server_url}/messages",
                json={"user": user, "text": text},
                timeout=5
            )
            return response.status_code == 201
        except:
            return False

# Create client
client = LongPollingClient("http://localhost:5002")
print("✅ Long polling client created!")

## 🧪 Experiment: See Long Polling in Action!

Let's see how the server holds the request open until data arrives.

In [ ]:
# First, let's see how long the server waits when there's no data

print("🔄 Starting long poll (will wait for data or timeout)...")
print("   The server will hold this request open!\n")

# Set a short timeout for demo
client.timeout = 10
client.last_timestamp = time.time()  # Only get new messages

start = time.time()
result = client.long_poll_once()
elapsed = time.time() - start

print(f"⏱️  Request took: {elapsed:.2f} seconds")

if result.get("timeout"):
    print("⏰ Timeout! Server had nothing to send.")
else:
    print(f"📬 Got {len(result['messages'])} message(s)!")

In [ ]:
# Now let's see what happens when a message arrives during a long poll!
import threading
import time

def send_message_after_delay(delay):
    """Send a message after a delay."""
    time.sleep(delay)
    print(f"\n📤 [{datetime.now().strftime('%H:%M:%S')}] Sending message...")
    client.send_message("Alice", "Hello from a delayed send!")

# Reset timestamp
client.last_timestamp = time.time()

print(f"🔄 [{datetime.now().strftime('%H:%M:%S')}] Starting long poll...")
print("   A message will be sent in 3 seconds...\n")

# Start the message sender in a background thread
sender = threading.Thread(target=send_message_after_delay, args=(3,))
sender.start()

# Long poll (will receive the message when it arrives!)
start = time.time()
result = client.long_poll_once()
elapsed = time.time() - start

print(f"\n📬 [{datetime.now().strftime('%H:%M:%S')}] Long poll returned!")
print(f"⏱️  Wait time: {elapsed:.2f} seconds")

if result["messages"]:
    for msg in result["messages"]:
        print(f"   └─ {msg['user']}: {msg['text']}")
    print("\n✅ Message received almost instantly after being sent!")
else:
    print("   No messages (this shouldn't happen!)")

sender.join()

## 📊 The Latency Problem with High-Frequency Updates

Long polling has a subtle issue: after receiving a message, the client must make a **new request**. If messages arrive in quick succession, this introduces latency.

```
Timeline (100ms network latency):

0ms    Client sends request ──────────────────►
100ms  Request arrives at server

150ms  💬 Message 1 arrives at server
150ms  Server responds immediately ◄──────────
250ms  Client receives Message 1

160ms  💬 Message 2 arrives at server (10ms after Message 1!)
       But client doesn't know yet...

250ms  Client makes new request ──────────────►
350ms  Request arrives at server
350ms  Server sends Message 2 ◄───────────────
450ms  Client receives Message 2

Total latency for Message 2: 290ms (450ms - 160ms)
```

This is **worse** than the 100ms network latency!

In [ ]:
# Let's demonstrate this latency issue

def demonstrate_burst_latency():
    """
    Show what happens when multiple messages arrive in quick succession.
    """
    print("📊 Demonstrating long polling latency with burst messages")
    print("="*60)
    
    # Reset
    client.last_timestamp = time.time()
    
    def send_burst_messages():
        """Send 3 messages 100ms apart."""
        time.sleep(1)  # Wait for long poll to start
        
        for i in range(3):
            send_time = datetime.now().strftime('%H:%M:%S.%f')[:-3]
            print(f"📤 [{send_time}] Sending message {i+1}")
            client.send_message("Burst", f"Message {i+1}")
            time.sleep(0.1)  # 100ms between messages
    
    # Start sender
    sender = threading.Thread(target=send_burst_messages)
    sender.start()
    
    # Receive messages via long polling
    print("\n🔄 Long polling for messages...\n")
    
    received = 0
    while received < 3:
        result = client.long_poll_once()
        
        for msg in result["messages"]:
            recv_time = datetime.now().strftime('%H:%M:%S.%f')[:-3]
            print(f"📬 [{recv_time}] Received: {msg['text']}")
            received += 1
    
    sender.join()
    
    print("\n💡 Notice: Each message required a new long-poll request!")
    print("   This adds latency for rapid message sequences.")

demonstrate_burst_latency()

## ✅ Advantages of Long Polling

1. **Near real-time** - Much faster than simple polling
2. **Still just HTTP** - Works with existing infrastructure
3. **No special client libraries** - Standard HTTP client works
4. **Firewall friendly** - Just HTTP requests
5. **Stateless-ish** - Server doesn't need persistent state

## ❌ Disadvantages

1. **Latency for burst messages** - Must reconnect after each response
2. **Connection overhead** - Many held connections use resources
3. **Timeout complexity** - Need to handle timeouts gracefully
4. **Load balancer issues** - Long requests might be terminated
5. **Monitoring challenges** - Requests look "slow" in metrics

In [ ]:
# Check server stats to see how many clients are waiting

def check_server_stats():
    try:
        response = requests.get("http://localhost:5002/stats")
        stats = response.json()
        
        print("📊 Server Statistics")
        print("="*40)
        print(f"   Waiting clients: {stats['waiting_clients']}")
        print(f"   Total messages:  {stats['total_messages']}")
    except:
        print("❌ Could not fetch server stats")

check_server_stats()

## 🎯 When to Use Long Polling

Long polling is great for:

| Use Case | Why Long Polling Works |
|----------|----------------------|
| Payment status | Need to know ASAP when payment completes |
| Job completion | Background task finished notification |
| Infrequent updates | Not many messages, but need them fast |
| Simple setups | When WebSocket is overkill |
| Legacy systems | When WebSocket isn't supported |

### Don't use it when:

- High-frequency updates (chat with lots of messages)
- Bi-directional communication needed
- Very low latency required
- Many concurrent users (resource overhead)

## 🔧 Implementation Tips

### 1. Set Appropriate Timeouts

```python
# Server timeout should be less than client timeout
SERVER_TIMEOUT = 30  # seconds
CLIENT_TIMEOUT = 35  # slightly longer!

# Also configure your load balancer!
# nginx: proxy_read_timeout 60s;
```

### 2. Handle Reconnection

```python
while True:
    try:
        result = long_poll()
        process_messages(result)
    except Exception as e:
        # Wait before retrying on error
        time.sleep(1)
```

### 3. Use HTTP Keep-Alive

```python
# Reuse the connection for subsequent polls
session = requests.Session()
```

In [ ]:
# Let's implement a robust long polling loop

def robust_long_polling_loop(duration=15):
    """
    A production-ready long polling loop with error handling.
    """
    print(f"🔄 Starting robust long polling loop for {duration}s...")
    print("   (Send messages from another client to see them!)\n")
    
    client.last_timestamp = time.time()
    client.timeout = 10  # Short timeout for demo
    
    start_time = time.time()
    poll_count = 0
    error_count = 0
    messages_received = 0
    
    while time.time() - start_time < duration:
        try:
            poll_count += 1
            current_time = datetime.now().strftime('%H:%M:%S')
            
            result = client.long_poll_once()
            
            if result.get("error"):
                error_count += 1
                print(f"[{current_time}] ❌ Error - waiting 1s before retry")
                time.sleep(1)  # Backoff on error
                continue
            
            if result["messages"]:
                for msg in result["messages"]:
                    messages_received += 1
                    print(f"[{current_time}] 📬 {msg['user']}: {msg['text']}")
            elif result.get("timeout"):
                print(f"[{current_time}] ⏰ Timeout - reconnecting...")
            
        except KeyboardInterrupt:
            print("\n🛑 Stopped by user")
            break
    
    print(f"\n📊 Summary:")
    print(f"   Polls made: {poll_count}")
    print(f"   Messages received: {messages_received}")
    print(f"   Errors: {error_count}")

# Send a message during the loop
def send_test_messages():
    time.sleep(3)
    client.send_message("System", "Test message 1")
    time.sleep(5)
    client.send_message("System", "Test message 2")

# Start message sender
sender = threading.Thread(target=send_test_messages)
sender.start()

# Run the loop
robust_long_polling_loop(duration=15)

sender.join()

## 🧪 Quick Quiz

1. **Why does long polling have lower latency than simple polling?**

2. **What happens if a message arrives right after the client disconnects from a long poll?**

3. **You're building a payment status checker. Would you use simple polling or long polling?**

In [ ]:
# Quiz answers

print("📝 Quiz Answers")
print("="*50)
print("")
print("1. Long polling responds IMMEDIATELY when data is available.")
print("   Simple polling waits for the next poll interval.")
print("")
print("2. The client will get it on the NEXT long poll request.")
print("   There's a brief window where messages can be delayed.")
print("   (This is why SSE/WebSocket can be better!)")
print("")
print("3. LONG POLLING! You want to know immediately when the")
print("   payment completes. The server can respond the instant")
print("   the payment status changes.")

## 📚 Summary

### What We Learned:

1. **Long polling** = Server holds request until data is available
2. **Lower latency** than simple polling for infrequent updates
3. **Still uses HTTP** - no special infrastructure needed
4. **Latency issue** with rapid consecutive messages
5. **Good for**: Payment status, job completion, infrequent updates

### Comparison Table:

| Feature | Simple Polling | Long Polling |
|---------|---------------|---------------|
| Latency | Up to poll interval | Near real-time |
| Server resources | Low (quick responses) | Higher (held connections) |
| Implementation | Very simple | Slightly complex |
| Burst messages | Consistent latency | Can add latency |

### Interview Tips:

> "Long polling is a good upgrade from simple polling when we need faster updates but don't want to add WebSocket complexity. It's perfect for infrequent but time-sensitive updates like payment confirmations."

### Next Up: Server-Sent Events (SSE)

In the next notebook, we'll see how **SSE** solves the burst message problem by keeping a single connection open for multiple updates!